# 4 - Python + EViews

**Windows only** - EViews automation is COM.

A dated pandas index becomes a dated EViews workfile, and comes back as a
`DatetimeIndex`.

In [ ]:
%load_ext econenv

In [ ]:
%econ doctor eviews

## A quarterly frame

In [ ]:
import numpy as np, pandas as pd, econenv

idx = pd.period_range('2000Q1', periods=60, freq='Q').to_timestamp()
rng = np.random.default_rng(11)
q = pd.DataFrame({'x': rng.normal(size=60).cumsum()}, index=idx)
q['y'] = 1.5 + 0.8 * q.x + rng.normal(scale=0.3, size=60)
q.head()

## Push - this creates `create Q 2000Q1 2014Q4`

In [ ]:
%%eviews -i q
equation eq1.ls y c x

In [ ]:
%eviews_get @pagefreq

In [ ]:
%eviews_get @pagesmpl

## Read results back

`%eviews_get` adds the `=` prefix EViews requires for anything that is not a
series expression.

In [ ]:
eviews = econenv.engine('eviews')
for expr in ['eq1.@r2', 'eq1.@rbar2', 'eq1.@regobs', 'eq1.@dw',
             'eq1.@coefs(1)', 'eq1.@coefs(2)', 'eq1.@stderrs(2)']:
    print(f'{expr:20} {eviews.pull_scalar(expr)}')

## The page comes home with its dates

In [ ]:
back = %eviews_pull
back.head()

In [ ]:
print(type(back.index).__name__, '|', back.index[0], '->', back.index[-1])

## Errors carry EViews' own message and the failing line

In [ ]:
try:
    eviews.execute('create u 5\nnot_a_real_command')
except Exception as exc:
    print(exc)